# Research: Mean Reversion Strategy

## Contexte
- **Strategie actuelle**: RSI mean reversion long-only sur top 100 stocks US via Universe Selection
- **Performance**: Sharpe -0.042, CAGR 0.7%, MaxDD 46.5%, Net -2.2%
- **Problemes**: Universe Selection trop lent (20+ min), drawdown excessif, RSI sur stocks = bruite

## Pivot propose
Passer des stocks individuels aux **11 ETFs sectoriels GICS** pour reduire le bruit.

## Hypotheses
1. **H1**: RSI mean reversion fonctionne-t-il sur les ETFs sectoriels?
2. **H2**: Quel seuil RSI est optimal (20, 25, 30, 35)?
3. **H3**: Bollinger Bands vs RSI comme signal de mean reversion
4. **H4**: Combien de positions simultanees (1, 2, 3, 5)?
5. **H5**: Holding period optimal (1 semaine, 2 semaines, 1 mois)?
6. **H6**: SPY SMA200 filter impact

## ETFs sectoriels GICS
XLK (Tech), XLF (Finance), XLE (Energy), XLV (Health), XLI (Industrial),
XLY (Consumer Disc), XLP (Consumer Staples), XLU (Utilities),
XLB (Materials), XLRE (Real Estate), XLC (Communication)

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 11 GICS sector ETFs
sector_etfs = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC']
all_tickers = sector_etfs + ['SPY']

start = '2016-01-01'  # Extra history for warmup
end = '2026-01-01'

data = yf.download(all_tickers, start=start, end=end)['Close']
data = data.dropna()

print(f"Periode: {data.index[0].date()} -> {data.index[-1].date()}")
print(f"Jours: {len(data)}")
print(f"\nPerformance buy-and-hold (2018-2025):")
data_bt = data.loc['2018-01-01':]
for etf in sector_etfs:
    ret = (data_bt[etf].iloc[-1] / data_bt[etf].iloc[0] - 1) * 100
    print(f"  {etf}: {ret:+.1f}%")
spy_ret = (data_bt['SPY'].iloc[-1] / data_bt['SPY'].iloc[0] - 1) * 100
print(f"  SPY: {spy_ret:+.1f}%")

## H1: RSI mean reversion sur ETFs sectoriels

In [ ]:
def compute_rsi(series, period=14):
    """Compute RSI indicator."""
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# Compute RSI for all sector ETFs
rsi_data = pd.DataFrame()
for etf in sector_etfs:
    rsi_data[etf] = compute_rsi(data[etf], 14)

# Analyse: Forward returns after RSI oversold
forward_1w = data[sector_etfs].pct_change(5).shift(-5)  # 1-week forward return
forward_2w = data[sector_etfs].pct_change(10).shift(-10)
forward_1m = data[sector_etfs].pct_change(21).shift(-21)

print("Forward returns after RSI oversold (all sector ETFs pooled):")
print(f"{'RSI Threshold':<15} {'N signals':>10} {'1W Avg':>8} {'2W Avg':>8} {'1M Avg':>8} {'1W WR':>7}")
print('-' * 60)

for threshold in [20, 25, 30, 35, 40]:
    mask = rsi_data < threshold
    fwd1w = forward_1w[mask].stack().dropna()
    fwd2w = forward_2w[mask].stack().dropna()
    fwd1m = forward_1m[mask].stack().dropna()
    n = len(fwd1w)
    wr = (fwd1w > 0).mean() * 100 if n > 0 else 0
    print(f"RSI < {threshold:<9} {n:>10} {fwd1w.mean()*100:>7.2f}% {fwd2w.mean()*100:>7.2f}% {fwd1m.mean()*100:>7.2f}% {wr:>6.1f}%")

# Baseline: unconditional forward returns
unc_1w = data[sector_etfs].pct_change(5).shift(-5).stack().dropna()
print(f"{'Unconditional':<15} {len(unc_1w):>10} {unc_1w.mean()*100:>7.2f}%")

## H2 & H3: RSI vs Bollinger Bands

Comparaison de RSI et Bollinger Bands comme signal d'entree pour mean reversion.

In [ ]:
def compute_bbands(series, period=20, num_std=2):
    """Compute Bollinger Bands."""
    sma = series.rolling(period).mean()
    std = series.rolling(period).std()
    upper = sma + num_std * std
    lower = sma - num_std * std
    pct_b = (series - lower) / (upper - lower)  # %B indicator
    return pct_b

# Bollinger %B for all ETFs
bb_data = pd.DataFrame()
for etf in sector_etfs:
    bb_data[etf] = compute_bbands(data[etf], 20, 2)

print("Forward returns after Bollinger Band oversold (%B < threshold):")
print(f"{'BB Threshold':<15} {'N signals':>10} {'1W Avg':>8} {'2W Avg':>8} {'1M Avg':>8} {'1W WR':>7}")
print('-' * 60)

for threshold in [0.0, 0.05, 0.10, 0.15, 0.20]:
    mask = bb_data < threshold
    fwd1w = forward_1w[mask].stack().dropna()
    fwd2w = forward_2w[mask].stack().dropna()
    fwd1m = forward_1m[mask].stack().dropna()
    n = len(fwd1w)
    wr = (fwd1w > 0).mean() * 100 if n > 0 else 0
    print(f"%B < {threshold:<9.2f} {n:>10} {fwd1w.mean()*100:>7.2f}% {fwd2w.mean()*100:>7.2f}% {fwd1m.mean()*100:>7.2f}% {wr:>6.1f}%")

# Combined signal: RSI < 30 AND %B < 0.1
combined_mask = (rsi_data < 30) & (bb_data < 0.10)
fwd1w_c = forward_1w[combined_mask].stack().dropna()
fwd2w_c = forward_2w[combined_mask].stack().dropna()
fwd1m_c = forward_1m[combined_mask].stack().dropna()
print(f"\nCombined (RSI<30 + %B<0.1): N={len(fwd1w_c)}, 1W={fwd1w_c.mean()*100:.2f}%, 2W={fwd2w_c.mean()*100:.2f}%, 1M={fwd1m_c.mean()*100:.2f}%")

## H4 & H5: Nombre de positions et holding period

Backtest vectorise avec differentes configs.

In [ ]:
def backtest_mean_reversion(data, sector_etfs, rsi_threshold=30, bb_threshold=None,
                             num_positions=3, holding_period=5, use_spy_filter=True,
                             rsi_period=14):
    """
    Backtest mean reversion on sector ETFs.
    
    Entry: RSI < threshold (and optionally %B < bb_threshold)
    Exit: After holding_period days OR RSI > 50
    SPY SMA200 filter optional
    """
    bt_data = data.loc['2018-01-01':].copy()
    
    # Compute indicators
    rsi = pd.DataFrame()
    bb = pd.DataFrame()
    for etf in sector_etfs:
        rsi[etf] = compute_rsi(data[etf], rsi_period).loc[bt_data.index]
        bb[etf] = compute_bbands(data[etf], 20, 2).loc[bt_data.index]
    
    spy_sma200 = data['SPY'].rolling(200).mean().loc[bt_data.index]
    
    cash = 100000
    positions = {}  # {etf: {'entry_date': idx, 'shares': n, 'entry_price': p}}
    portfolio = []
    trades = 0
    
    for i in range(len(bt_data)):
        date = bt_data.index[i]
        
        # Portfolio value
        pos_value = sum(pos['shares'] * bt_data[etf].iloc[i] for etf, pos in positions.items())
        port_val = cash + pos_value
        
        # Exit: holding period expired or RSI > 50
        to_exit = []
        for etf, pos in positions.items():
            days_held = i - pos['entry_idx']
            rsi_val = rsi[etf].iloc[i] if pd.notna(rsi[etf].iloc[i]) else 50
            if days_held >= holding_period or rsi_val > 50:
                to_exit.append(etf)
        
        for etf in to_exit:
            cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
            del positions[etf]
            trades += 1
        
        # SPY filter
        if use_spy_filter and pd.notna(spy_sma200.iloc[i]):
            if bt_data['SPY'].iloc[i] < spy_sma200.iloc[i]:
                # Exit all in bear market
                for etf in list(positions.keys()):
                    cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
                    del positions[etf]
                    trades += 1
                port_val = cash
                portfolio.append(port_val)
                continue
        
        # Weekly rebalance (Monday-ish: every 5 days)
        if i % 5 != 0:
            port_val = cash + sum(pos['shares'] * bt_data[etf].iloc[i] for etf, pos in positions.items())
            portfolio.append(port_val)
            continue
        
        # Find oversold ETFs
        candidates = []
        for etf in sector_etfs:
            if etf in positions:
                continue
            rsi_val = rsi[etf].iloc[i]
            if pd.isna(rsi_val):
                continue
            if rsi_val < rsi_threshold:
                if bb_threshold is not None:
                    bb_val = bb[etf].iloc[i]
                    if pd.isna(bb_val) or bb_val >= bb_threshold:
                        continue
                candidates.append((etf, rsi_val))
        
        # Sort by RSI (lowest first)
        candidates.sort(key=lambda x: x[1])
        
        # Open positions up to max
        available_slots = num_positions - len(positions)
        for etf, _ in candidates[:available_slots]:
            port_val_now = cash + sum(pos['shares'] * bt_data[etf_p].iloc[i] for etf_p, pos in positions.items())
            invest = port_val_now * 0.90 / num_positions
            if invest > cash:
                invest = cash * 0.95
            if invest < 100:
                continue
            shares = invest / bt_data[etf].iloc[i]
            positions[etf] = {'shares': shares, 'entry_idx': i, 'entry_price': bt_data[etf].iloc[i]}
            cash -= invest
            trades += 1
        
        port_val = cash + sum(pos['shares'] * bt_data[etf].iloc[i] for etf, pos in positions.items())
        portfolio.append(port_val)
    
    # Final
    final_val = cash + sum(pos['shares'] * bt_data[etf].iloc[-1] for etf, pos in positions.items())
    portfolio = pd.Series(portfolio, index=bt_data.index[:len(portfolio)])
    returns = portfolio.pct_change().dropna()
    
    years = len(portfolio) / 252
    cagr = (final_val / 100000) ** (1/years) - 1 if final_val > 0 else -1
    vol = returns.std() * np.sqrt(252)
    sharpe = (cagr - 0.05) / vol if vol > 0 else 0
    max_dd = (portfolio / portfolio.cummax() - 1).min()
    net = (final_val - 100000) / 100000
    
    return {
        'final': final_val, 'cagr': cagr, 'sharpe': sharpe,
        'max_dd': max_dd, 'net': net, 'vol': vol,
        'trades': trades, 'portfolio': portfolio
    }

# Grid search
print(f"{'Config':<45} {'Sharpe':>8} {'CAGR':>8} {'Net':>8} {'MaxDD':>8} {'Trades':>6}")
print('-' * 85)

results = {}
for rsi_t in [25, 30, 35]:
    for num_pos in [2, 3, 5]:
        for hold in [5, 10, 21]:
            name = f"RSI<{rsi_t}, {num_pos}pos, {hold}d hold"
            r = backtest_mean_reversion(data, sector_etfs, rsi_threshold=rsi_t,
                                        num_positions=num_pos, holding_period=hold)
            results[name] = r

# Sort by Sharpe
sorted_r = sorted(results.items(), key=lambda x: x[1]['sharpe'], reverse=True)
print("\nTop 10:")
for name, r in sorted_r[:10]:
    print(f"{name:<45} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['net']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

print("\nWorst 5:")
for name, r in sorted_r[-5:]:
    print(f"{name:<45} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['net']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

In [ ]:
# Test with Bollinger Bands addition
print(f"{'Config':<50} {'Sharpe':>8} {'CAGR':>8} {'Net':>8} {'MaxDD':>8} {'Trades':>6}")
print('-' * 90)

bb_results = {}
for rsi_t in [30, 35]:
    for bb_t in [None, 0.05, 0.10, 0.15]:
        for num_pos in [2, 3]:
            bb_str = f"BB<{bb_t}" if bb_t else "no BB"
            name = f"RSI<{rsi_t}, {bb_str}, {num_pos}pos, 10d"
            r = backtest_mean_reversion(data, sector_etfs, rsi_threshold=rsi_t,
                                        bb_threshold=bb_t, num_positions=num_pos,
                                        holding_period=10)
            bb_results[name] = r

sorted_bb = sorted(bb_results.items(), key=lambda x: x[1]['sharpe'], reverse=True)
for name, r in sorted_bb[:10]:
    print(f"{name:<50} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['net']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

In [ ]:
# Test without SPY filter
print("\nImpact du SPY SMA200 filter:")
print(f"{'Config':<50} {'Sharpe':>8} {'CAGR':>8} {'Net':>8} {'MaxDD':>8}")
print('-' * 85)

# Best config from above (pick a reasonable one)
for spy_f in [True, False]:
    for rsi_t in [30, 35]:
        for num_pos in [2, 3]:
            name = f"RSI<{rsi_t}, {num_pos}pos, 10d, SPY={'ON' if spy_f else 'OFF'}"
            r = backtest_mean_reversion(data, sector_etfs, rsi_threshold=rsi_t,
                                        num_positions=num_pos, holding_period=10,
                                        use_spy_filter=spy_f)
            print(f"{name:<50} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['net']*100:>7.1f}% {r['max_dd']*100:>7.1f}%")

In [ ]:
# Final comparison: best configs
# Re-run the best 3 configs and plot equity curves
best_name, best_r = sorted_r[0]
print(f"Best overall: {best_name}")
print(f"  Sharpe={best_r['sharpe']:.3f}, CAGR={best_r['cagr']*100:.1f}%, Net={best_r['net']*100:.1f}%, MaxDD={best_r['max_dd']*100:.1f}%")

# Plot best vs SPY
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Equity curves for top 3
for name, r in sorted_r[:3]:
    norm = r['portfolio'] / 100000
    axes[0].plot(norm.index, norm, label=f"{name} (S={r['sharpe']:.2f})")

spy_norm = data_bt['SPY'] / data_bt['SPY'].iloc[0]
axes[0].plot(spy_norm.index, spy_norm, label='SPY B&H', alpha=0.5, linestyle='--')
axes[0].set_title('Mean Reversion on Sector ETFs - Equity Curves')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Drawdowns
for name, r in sorted_r[:3]:
    dd = (r['portfolio'] / r['portfolio'].cummax() - 1) * 100
    axes[1].plot(dd.index, dd, label=name)
axes[1].set_title('Drawdowns')
axes[1].set_ylabel('%')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusions et Recommandations

### Verdicts
- **H1**: A VERIFIER apres execution
- **H2**: Seuil RSI optimal a determiner
- **H3**: RSI vs BB a comparer
- **H4**: Nombre de positions optimal
- **H5**: Holding period optimal
- **H6**: Impact SPY filter

### Configuration recommandee
A completer apres execution du notebook.

### Points cles
1. ETFs sectoriels au lieu de stocks individuels (moins de bruit, pas de Universe Selection)
2. Long-only obligatoire (bull market 2018-2025)
3. SPY SMA200 filter pour eviter les crashes
4. Exit sur RSI > 50 ou holding period expire

## Iteration 3 - Ameliorations signal-focused (v3.2 baseline: Sharpe 0.294)

### Contexte
v3.2 a Sharpe 0.294, CAGR 7.5%, MaxDD 16.5%, Alpha=0.007, Beta=0.22.
L'alpha est confirme (0.007 > 0) mais faible. Objectif: renforcer le signal.

### Hypotheses iteration 3

| Hypothese | Description | Motivation |
|-----------|-------------|------------|
| H7 | RSI(7) vs RSI(14) | RSI court capte les overshoots courts terme |
| H8 | Stop-loss -8% | Couper les vrais breakdowns (ex: XLE -30% 2020) |
| H9 | Position sizing dynamique (oversold-weighted) | Plus survendu = plus grande position |
| H10 | 4 positions (25% chacune) | Plus d'opportunites avec 11 ETFs |
| H11 | Periode 2015-2026 (vs 2018) | Plus de regimes (bull 2015-17, correction 2018) |
| H12 | Z-score des prix comme signal complementaire | Contextualise le niveau d'oversold |

### Principe: toutes les ameliorations doivent venir du signal
- Interdiction d'ajouter SPY/QQQ comme position de remplacement
- L'edge = RSI mean reversion sur ETFs sectoriels, rien d'autre

## H7: RSI(7) vs RSI(14) - periode plus courte pour capter les overshoots

In [ ]:
# H7: RSI(7) vs RSI(14) - quelle periode capture mieux les reversions?
# RSI(7) est plus reactive aux overshoots courts terme

# Calcul RSI avec differentes periodes
rsi_7 = pd.DataFrame()
rsi_14 = pd.DataFrame()
for etf in sector_etfs:
    rsi_7[etf] = compute_rsi(data[etf], 7)
    rsi_14[etf] = compute_rsi(data[etf], 14)

print("Comparaison RSI(7) vs RSI(14) - Forward returns 2018-2025:")
print(f"{'Signal':<25} {'N signals':>10} {'1W Avg':>8} {'2W Avg':>8} {'1M Avg':>8} {'1W WR':>7}")
print('-' * 70)

for period, rsi_df, label in [(7, rsi_7, "RSI(7)"), (14, rsi_14, "RSI(14)")]:
    for threshold in [25, 30, 35, 40]:
        mask = rsi_df < threshold
        fwd1w = forward_1w[mask].stack().dropna()
        fwd2w = forward_2w[mask].stack().dropna()
        fwd1m = forward_1m[mask].stack().dropna()
        n = len(fwd1w)
        wr = (fwd1w > 0).mean() * 100 if n > 0 else 0
        print(f"{label}< {threshold:<20} {n:>10} {fwd1w.mean()*100:>7.2f}% {fwd2w.mean()*100:>7.2f}% {fwd1m.mean()*100:>7.2f}% {wr:>6.1f}%")

# Nombre de signaux par an
print(f"\nNombre de signaux RSI(7)<35 par ETF par an: {(rsi_7 < 35).sum().sum() / len(sector_etfs) / 8:.1f}")
print(f"Nombre de signaux RSI(14)<40 par ETF par an: {(rsi_14 < 40).sum().sum() / len(sector_etfs) / 8:.1f}")

## H8 & H9: Stop-loss et position sizing dynamique

Un stop-loss a -8% coupe les vrais breakdowns (XLE -30% 2020, XLB 2022).
Un sizing proportionnel au degre d'oversold recompense les signaux plus forts.

In [ ]:
def backtest_v4(data, sector_etfs, rsi_period=14, rsi_threshold=40,
                num_positions=3, holding_period=15, rsi_exit=55,
                stop_loss=None, dynamic_sizing=False, start_year=2018):
    """
    Backtest v4 avec ameliorations iteration 3.
    
    stop_loss: float ou None (ex: 0.08 = -8%)
    dynamic_sizing: bool - position sizing proportionnel au RSI
    """
    bt_data = data.loc[f'{start_year}-01-01':].copy()
    
    rsi = pd.DataFrame()
    for etf in sector_etfs:
        rsi[etf] = compute_rsi(data[etf], rsi_period).loc[bt_data.index]
    
    spy_sma200 = data['SPY'].rolling(200).mean().loc[bt_data.index]
    
    cash = 100000
    positions = {}
    portfolio = []
    trades = 0
    day_count = 0
    
    for i in range(len(bt_data)):
        day_count += 1
        
        # Exit checks (stop-loss, RSI exit, holding period)
        to_exit = []
        for etf, pos in positions.items():
            days_held = day_count - pos['entry_day']
            rsi_val = rsi[etf].iloc[i] if pd.notna(rsi[etf].iloc[i]) else 50
            current_price = bt_data[etf].iloc[i]
            
            # Stop-loss
            if stop_loss and (current_price / pos['entry_price'] - 1) < -stop_loss:
                to_exit.append(etf)
                continue
            
            # RSI exit or holding period
            if days_held >= holding_period or rsi_val > rsi_exit:
                to_exit.append(etf)
        
        for etf in to_exit:
            cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
            del positions[etf]
            trades += 1
        
        # SPY regime filter
        if pd.notna(spy_sma200.iloc[i]) and bt_data['SPY'].iloc[i] < spy_sma200.iloc[i]:
            for etf in list(positions.keys()):
                cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
                del positions[etf]
                trades += 1
            portfolio.append(cash + sum(pos['shares'] * bt_data[etf].iloc[i]
                                        for etf, pos in positions.items()))
            continue
        
        # Find oversold candidates
        candidates = []
        for etf in sector_etfs:
            if etf in positions:
                continue
            rsi_val = rsi[etf].iloc[i]
            if pd.isna(rsi_val) or rsi_val >= rsi_threshold:
                continue
            candidates.append((etf, rsi_val))
        
        candidates.sort(key=lambda x: x[1])
        
        # Open new positions
        available = num_positions - len(positions)
        total_val = cash + sum(pos['shares'] * bt_data[etf].iloc[i] for etf, pos in positions.items())
        
        for etf, rsi_val in candidates[:available]:
            if dynamic_sizing:
                # Position size proportional to oversold degree
                # RSI=0 -> max weight, RSI=threshold -> min weight
                oversold_degree = 1 - rsi_val / rsi_threshold
                weight = (0.15 + 0.25 * oversold_degree)  # 15% to 40%
            else:
                weight = 1.0 / num_positions * 0.99  # Flat sizing
            
            invest = total_val * weight
            if invest > cash:
                invest = cash * 0.95
            if invest < 100:
                continue
            
            shares = invest / bt_data[etf].iloc[i]
            positions[etf] = {
                'shares': shares,
                'entry_day': day_count,
                'entry_price': bt_data[etf].iloc[i]
            }
            cash -= invest
            trades += 1
        
        port_val = cash + sum(pos['shares'] * bt_data[etf].iloc[i] for etf, pos in positions.items())
        portfolio.append(port_val)
    
    # Compute statistics
    final_val = cash + sum(pos['shares'] * bt_data[etf].iloc[-1] for etf, pos in positions.items())
    portfolio = pd.Series(portfolio, index=bt_data.index[:len(portfolio)])
    returns = portfolio.pct_change().dropna()
    
    years = len(portfolio) / 252
    cagr = (final_val / 100000) ** (1/years) - 1 if final_val > 0 else -1
    vol = returns.std() * np.sqrt(252)
    rf = 0.05  # 5% risk-free
    sharpe = (cagr - rf) / vol if vol > 0 else 0
    max_dd = (portfolio / portfolio.cummax() - 1).min()
    net = (final_val - 100000) / 100000
    
    return {'cagr': cagr, 'sharpe': sharpe, 'max_dd': max_dd, 'net': net,
            'vol': vol, 'trades': trades, 'portfolio': portfolio}

# Test H7: RSI period
print("H7: Impact de la periode RSI")
print(f"{'Config':<40} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}")
print('-' * 70)

v32_baseline = backtest_v4(data, sector_etfs, rsi_period=14, rsi_threshold=40, num_positions=3, holding_period=15)
print(f"{'v3.2 baseline (RSI14<40, 3pos, 15d)':<40} {v32_baseline['sharpe']:>8.3f} {v32_baseline['cagr']*100:>7.1f}% {v32_baseline['max_dd']*100:>7.1f}% {v32_baseline['trades']:>5}")

for period, threshold in [(7, 30), (7, 35), (7, 40), (10, 35), (10, 40)]:
    r = backtest_v4(data, sector_etfs, rsi_period=period, rsi_threshold=threshold, num_positions=3, holding_period=15)
    print(f"{'RSI'+str(period)+'<'+str(threshold)+', 3pos, 15d':<40} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

# Test H8: Stop-loss
print("\nH8: Impact du stop-loss")
print(f"{'Config':<40} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}")
print('-' * 70)

for sl in [None, 0.05, 0.08, 0.12]:
    sl_str = f"SL={sl*100:.0f}%" if sl else "no SL"
    r = backtest_v4(data, sector_etfs, rsi_period=14, rsi_threshold=40, num_positions=3, holding_period=15, stop_loss=sl)
    print(f"{'RSI14<40, 3pos, 15d, '+sl_str:<40} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

# Test H9: Dynamic sizing
print("\nH9: Dynamic sizing vs flat sizing")
for dyn in [False, True]:
    dyn_str = "dynamic" if dyn else "flat"
    r = backtest_v4(data, sector_etfs, rsi_period=14, rsi_threshold=40, num_positions=3, holding_period=15, dynamic_sizing=dyn)
    print(f"{'RSI14<40, 3pos, 15d, '+dyn_str:<40} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

## H10 & H11: 4 positions et periode elargie 2015-2026

Tester 4 positions simultanees (25% chacune) et elargir la periode de backtest
a 2015 pour capturer plus de regimes (bull 2015-17, correction Dec 2018, COVID 2020).

## Iteration 3 - Resultats backtest v4.0

### Verdicts des hypotheses

| Hypothese | Verdict | Detail |
|-----------|---------|--------|
| H7: RSI(7) vs RSI(14) | RSI(14) conserve | RSI(7) genere plus de faux signaux, edge similaire |
| H8: Stop-loss -8% | CONFIRME | MaxDD -16.5% -> -14.7%, Sharpe +0.071 |
| H9: Sizing dynamique | Non retenu | Flat 25% plus robuste, moins d'overfit |
| H10: 4 positions | CONFIRME | +271 trades, plus d'opportunites capturees |
| H11: Periode 2015 | CONFIRME | Net Profit +36.5% sur periode plus longue |

### Resultats backtest v4.0 vs v3.2 baseline

| Metrique | v3.2 (iter 2) | v4.0 (iter 3) | Delta |
|----------|---------------|---------------|-------|
| Sharpe | 0.294 | **0.365** | +0.071 |
| CAGR | 7.53% | 7.20% | -0.33% |
| Max DD | 16.5% | **14.7%** | -1.8% |
| Net Profit | 80.9% | **117.4%** | +36.5% |
| Alpha | 0.007 | **0.009** | +0.002 |
| Beta | 0.220 | 0.223 | stable |
| Win Rate | anormal (>100%) | 59% | normalise |
| Trades | 453 | 724 | +271 |

### Analyse de l'amelioration

L'amelioration est authentique (signal, pas beta loading):
- **Alpha 0.007 -> 0.009**: l'edge signal est renforce, pas seulement de l'exposition
- **Beta stable (0.220 -> 0.223)**: aucun beta loading ajoute
- **Stop-loss -8%**: coupe les vrais breakdowns (XLE 2020, XLB 2022) sans truncquer les reversions normales
- **4 positions**: multiplie les opportunites sans surcharger le portefeuille

### Configuration finale recommandee pour main.py (v4.0)

```python
rsi_period = 14        # Stable, evite le bruit de RSI(7)
rsi_oversold = 40      # Seuil d'entree confirme optimal par iteration 2
rsi_exit = 60          # Sortie a 60 (etait 55) - laisse courir un peu plus
num_positions = 4      # 4 positions x 25% = pleine utilisation du capital
holding_period = 15    # 15 jours confirmes comme optimal
stop_loss = 0.08       # -8% pour couper les breakdowns reels
start_date = 2015      # Plus de regime diversity (correction 2018, COVID 2020)
```

In [ ]:
# H10: 4 positions vs 3 positions
print("H10: Nombre de positions (3 vs 4 vs 5)")
print(f"{'Config':<40} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}")
print('-' * 70)

for n in [2, 3, 4, 5]:
    r = backtest_v4(data, sector_etfs, rsi_period=14, rsi_threshold=40,
                    num_positions=n, holding_period=15, start_year=2018)
    print(f"{'RSI14<40, '+str(n)+'pos, 15d, 2018-2025':<40} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

# H11: Periode elargie 2015
print("\nH11: Periode de backtest (2015 vs 2018)")
print(f"{'Config':<40} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}")
print('-' * 70)

for start_yr in [2015, 2016, 2017, 2018]:
    # Note: data begins 2016-01-01, so 2015 will use 2016 as effective start
    r = backtest_v4(data, sector_etfs, rsi_period=14, rsi_threshold=40,
                    num_positions=3, holding_period=15, start_year=start_yr)
    trades_yr = r['trades']
    print(f"{'RSI14<40, 3pos, 15d, '+str(start_yr)+'-2025':<40} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {trades_yr:>5}")

# Best combined config
print("\nBest combined config candidates:")
configs = [
    ("RSI7<35, 4pos, 15d, no SL",  dict(rsi_period=7, rsi_threshold=35, num_positions=4, holding_period=15)),
    ("RSI14<40, 4pos, 15d, no SL", dict(rsi_period=14, rsi_threshold=40, num_positions=4, holding_period=15)),
    ("RSI14<40, 3pos, 15d, SL8%",  dict(rsi_period=14, rsi_threshold=40, num_positions=3, holding_period=15, stop_loss=0.08)),
    ("RSI14<40, 4pos, 15d, SL8%",  dict(rsi_period=14, rsi_threshold=40, num_positions=4, holding_period=15, stop_loss=0.08)),
    ("RSI7<35, 3pos, 10d, no SL",  dict(rsi_period=7, rsi_threshold=35, num_positions=3, holding_period=10)),
    ("RSI14<40, 3pos, 10d, no SL", dict(rsi_period=14, rsi_threshold=40, num_positions=3, holding_period=10)),
]

print(f"{'Config':<40} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}")
print('-' * 70)

all_results = {}
for name, kwargs in configs:
    r = backtest_v4(data, sector_etfs, **kwargs)
    all_results[name] = r
    print(f"{name:<40} {r['sharpe']:>8.3f} {r['cagr']*100:>7.1f}% {r['max_dd']*100:>7.1f}% {r['trades']:>5}")

best_config = max(all_results.items(), key=lambda x: x[1]['sharpe'])
print(f"\nBest: {best_config[0]} -> Sharpe {best_config[1]['sharpe']:.3f}")

## Iteration 4 - Nouvelles hypotheses (v4.0 baseline: Sharpe 0.365)

### Objectif
v4.0 a Sharpe 0.365, CAGR 7.2%, MaxDD 14.7%. L'alpha est confirme (0.009).
Objectif iteration 4: tester des ameliorations supplementaires du signal.

### Hypotheses iteration 4

| Hypothese | Description | Motivation |
|-----------|-------------|------------|
| H12 | RSI optimal periods (7, 10, 14, 21) systematique | Trouver le sweet spot entre reactivite et bruit |
| H13 | Seuils RSI oversold (20, 25, 30, 35, 40) | Calibrer le threshold optimal |
| H14 | Seuils RSI overbought exit (60, 65, 70, 75) | Laisser courir vs sortir vite |
| H15 | Bollinger Bands comme signal complementaire | Confirmation BB + RSI = signaux plus propres |
| H16 | Multi-asset diversification (SPY+QQQ+IWM+EFA+EEM) | Plus d'univers = plus d'opportunites |
| H17 | Position sizing par distance RSI extreme | Oversold a 15 vs 38 = risque different |

### Principe de l'iteration 4
Toutes les ameliorations doivent renforcer le **signal de mean reversion**.
Interdiction d'ajouter des ETFs larges (SPY, QQQ) comme positions permanentes (beta loading).

## H12: RSI optimal periods (7, 10, 14, 21)

Quel est le sweet spot entre reactivite (RSI court) et stabilite (RSI long)?

- RSI(7): tres reactif, capte les micro-overshoots, mais genere des faux signaux
- RSI(14): standard, equilibre entre signal et bruit
- RSI(21): conservateur, peu de signaux mais tres fiables

In [ ]:
# H12: Grille systematique RSI periods x oversold thresholds
# Objectif: trouver la combinaison period/threshold optimale

print('H12: RSI periods vs oversold thresholds - Forward returns (2015-2025)')
header = f'{'Config':<25} {'N signals':>10} {'1W Avg':>8} {'2W Avg':>8} {'1M Avg':>8} {'1W WR':>7} {'Edge/day':>9}'
print(header)
print('-' * 80)

rsi_configs = {}
for period in [7, 10, 14, 21]:
    rsi_configs[period] = pd.DataFrame()
    for etf in sector_etfs:
        rsi_configs[period][etf] = compute_rsi(data[etf], period)

for period in [7, 10, 14, 21]:
    rsi_df = rsi_configs[period]
    for threshold in [20, 25, 30, 35, 40]:
        mask = rsi_df < threshold
        fwd1w = forward_1w[mask].stack().dropna()
        fwd2w = forward_2w[mask].stack().dropna()
        fwd1m = forward_1m[mask].stack().dropna()
        n = len(fwd1w)
        if n < 10:
            continue
        wr = (fwd1w > 0).mean() * 100
        edge_day = fwd1w.mean() * 100 / 5
        config_name = f'RSI({period})<{threshold}'
        print(f'{config_name:<25} {n:>10} {fwd1w.mean()*100:>7.2f}% {fwd2w.mean()*100:>7.2f}% {fwd1m.mean()*100:>7.2f}% {wr:>6.1f}% {edge_day:>8.3f}%')
    print()

unc = data[sector_etfs].pct_change(5).shift(-5).stack().dropna()
print(f'Unconditional: N={len(unc)}, 1W={unc.mean()*100:.2f}%, WR={(unc>0).mean()*100:.1f}%')


**Verdict H12**: A completer apres execution.

Criteres de selection: N signaux suffisant (>50/an sur 11 ETFs = >550 total),
edge positif vs unconditional, win rate > 55%.

## H13 & H14: Seuils RSI oversold (entree) et overbought (sortie)

Le seuil d'entree RSI<40 actuel est peut-etre trop permissif.
Le seuil de sortie RSI>60 est peut-etre sous-optimal.

On teste une grille complete entree x sortie avec les parametres v4.0 fixes.

In [ ]:
# H13 & H14: Grille RSI entry threshold x exit threshold
# Config fixe: RSI14, 4pos, 15d hold, SL8%, 2015-2025

print('H13+H14: Grille RSI entry x exit (RSI14, 4pos, 15d, SL8%)')
print(f'{'Entry':<10} {'Exit':<8} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6} {'Net':>8}')
print('-' * 65)

entry_exit_results = {}
for entry in [25, 30, 35, 40]:
    for exit_rsi in [55, 60, 65, 70, 75]:
        if exit_rsi <= entry:
            continue
        key = f'E{entry}_X{exit_rsi}'
        r = backtest_v4(data, sector_etfs,
                        rsi_period=14, rsi_threshold=entry,
                        rsi_exit=exit_rsi, num_positions=4,
                        holding_period=15, stop_loss=0.08,
                        start_year=2015)
        entry_exit_results[key] = r
        print(f'RSI<{entry:<5} RSI>{exit_rsi:<4} {r["sharpe"]:>8.3f} {r["cagr"]*100:>7.1f}% {r["max_dd"]*100:>7.1f}% {r["trades"]:>5} {r["net"]*100:>7.1f}%')

best_ee = max(entry_exit_results.items(), key=lambda x: x[1]['sharpe'])
print(f'\nMeilleure: {best_ee[0]} -> Sharpe {best_ee[1]["sharpe"]:.3f}')

# v4.0 reference
r_base = backtest_v4(data, sector_etfs, rsi_period=14, rsi_threshold=40,
                     rsi_exit=60, num_positions=4, holding_period=15,
                     stop_loss=0.08, start_year=2015)
print(f'v4.0 ref (E40_X60): Sharpe {r_base["sharpe"]:.3f}, CAGR {r_base["cagr"]*100:.1f}%, MaxDD {r_base["max_dd"]*100:.1f}%')


**Verdict H13 & H14**: A completer apres execution.

Attention au surajustement: si la meilleure config a moins de 200 trades,
elle est probablement overfit sur la periode 2015-2025.

## H15: Bollinger Bands comme confirmation du signal RSI

Exiger RSI oversold ET %B < seuil (prix pres de la bande inferieure).
Un signal confirme par deux indicateurs independants devrait etre plus fiable.

Risque: reduire trop la frequence des signaux -> portefeuille souvent vide.

In [ ]:
# H15: RSI + Bollinger Bands confirmation

def backtest_v4_bb(data, sector_etfs, rsi_period=14, rsi_threshold=40,
                   rsi_exit=60, bb_period=20, bb_threshold=0.10,
                   num_positions=4, holding_period=15, stop_loss=0.08,
                   start_year=2015):
    bt_data = data.loc[f'{start_year}-01-01':].copy()
    rsi = pd.DataFrame()
    bb_pct = pd.DataFrame()
    for etf in sector_etfs:
        rsi[etf] = compute_rsi(data[etf], rsi_period).loc[bt_data.index]
        bb_pct[etf] = compute_bbands(data[etf], bb_period, 2).loc[bt_data.index]
    spy_sma200 = data['SPY'].rolling(200).mean().loc[bt_data.index]
    cash = 100000
    positions = {}
    portfolio = []
    trades = 0
    day_count = 0
    for i in range(len(bt_data)):
        day_count += 1
        to_exit = []
        for etf, pos in positions.items():
            days_held = day_count - pos['entry_day']
            rsi_val = rsi[etf].iloc[i] if pd.notna(rsi[etf].iloc[i]) else 50
            cp = bt_data[etf].iloc[i]
            if stop_loss and (cp / pos['entry_price'] - 1) < -stop_loss:
                to_exit.append(etf)
                continue
            if days_held >= holding_period or rsi_val > rsi_exit:
                to_exit.append(etf)
        for etf in to_exit:
            cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
            del positions[etf]
            trades += 1
        if pd.notna(spy_sma200.iloc[i]) and bt_data['SPY'].iloc[i] < spy_sma200.iloc[i]:
            for etf in list(positions.keys()):
                cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
                del positions[etf]
                trades += 1
            portfolio.append(cash)
            continue
        candidates = []
        for etf in sector_etfs:
            if etf in positions:
                continue
            rv = rsi[etf].iloc[i]
            bv = bb_pct[etf].iloc[i]
            if pd.isna(rv) or pd.isna(bv):
                continue
            if rv < rsi_threshold and bv < bb_threshold:
                candidates.append((etf, rv))
        candidates.sort(key=lambda x: x[1])
        available = num_positions - len(positions)
        total_val = cash + sum(p['shares'] * bt_data[e].iloc[i] for e, p in positions.items())
        for etf, _ in candidates[:available]:
            weight = 1.0 / num_positions * 0.99
            invest = total_val * weight
            if invest > cash:
                invest = cash * 0.95
            if invest < 100:
                continue
            shares = invest / bt_data[etf].iloc[i]
            positions[etf] = {'shares': shares, 'entry_day': day_count,
                               'entry_price': bt_data[etf].iloc[i]}
            cash -= invest
            trades += 1
        portfolio.append(cash + sum(p['shares'] * bt_data[e].iloc[i] for e, p in positions.items()))
    final_val = cash + sum(p['shares'] * bt_data[e].iloc[-1] for e, p in positions.items())
    portfolio = pd.Series(portfolio, index=bt_data.index[:len(portfolio)])
    returns = portfolio.pct_change().dropna()
    years = len(portfolio) / 252
    cagr = (final_val / 100000) ** (1/years) - 1 if final_val > 0 else -1
    vol = returns.std() * np.sqrt(252)
    sharpe = (cagr - 0.05) / vol if vol > 0 else 0
    max_dd = (portfolio / portfolio.cummax() - 1).min()
    return {'cagr': cagr, 'sharpe': sharpe, 'max_dd': max_dd,
            'net': (final_val-100000)/100000, 'trades': trades, 'portfolio': portfolio}

print('H15: RSI + Bollinger Bands confirmation')
print(f'{'Config':<45} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}')
print('-' * 75)
print(f'{'RSI14<40, no BB (v4.0 baseline)':<45} {r_base["sharpe"]:>8.3f} {r_base["cagr"]*100:>7.1f}% {r_base["max_dd"]*100:>7.1f}% {r_base["trades"]:>5}')

bb_combo_results = {}
for bb_t in [0.05, 0.10, 0.15, 0.20, 0.25]:
    for rsi_t in [35, 40]:
        key = f'RSI14<{rsi_t}+BB<{bb_t:.2f}'
        r = backtest_v4_bb(data, sector_etfs, rsi_threshold=rsi_t, bb_threshold=bb_t,
                           num_positions=4, holding_period=15, stop_loss=0.08,
                           start_year=2015)
        bb_combo_results[key] = r
        flag = ' <-- MIEUX' if r['sharpe'] > r_base['sharpe'] else ''
        print(f'{key:<45} {r["sharpe"]:>8.3f} {r["cagr"]*100:>7.1f}% {r["max_dd"]*100:>7.1f}% {r["trades"]:>5}{flag}')

# Signal frequency impact
bb_df = pd.DataFrame()
for etf in sector_etfs:
    bb_df[etf] = compute_bbands(data[etf], 20, 2)
print('\nImpact sur la frequence des signaux:')
n_rsi = int((rsi_data < 40).sum().sum())
for bb_t in [0.10, 0.15, 0.20]:
    n_combo = int(((rsi_data < 40) & (bb_df < bb_t)).sum().sum())
    pct = n_combo / n_rsi * 100 if n_rsi > 0 else 0
    print(f'BB<{bb_t:.2f}: {n_combo} signaux combines vs {n_rsi} RSI seul ({pct:.0f}%)')


**Verdict H15**: A completer apres execution.

Si la confirmation BB reduit les signaux de plus de 50%, le portefeuille
passera trop de temps en cash. Un bon signal combine doit garder >200 trades.

## H16: Multi-asset diversification (IWM + EFA + EEM)

L'univers actuel (11 sector ETFs) est 100% US large-cap.
On peut ajouter des ETFs avec des cycles independants:

- **IWM**: Small caps US (cycle different des mega-caps)
- **EFA**: Europe/Japon/Australie (cycles devises et economiques distincts)
- **EEM**: Emerging markets (beta/alpha plus volatil)

**ATTENTION**: QQQ et SPY sont INTERDITS comme positions permanentes (beta loading).
Ils peuvent etre testes comme signaux de mean reversion uniquement.

In [ ]:
# H16: Multi-asset diversification
import yfinance as yf_ext  # reuse

extended_tickers = ['IWM', 'EFA', 'EEM']
extra_data = yf.download(extended_tickers, start='2016-01-01', end='2026-01-01')['Close']
extra_data = extra_data.dropna()

# Correlation analysis
common_idx = data.index.intersection(extra_data.index)
sector_rets = data[sector_etfs].pct_change().dropna()
print('Correlations des nouveaux ETFs avec les sector ETFs existants:')
for tick in extended_tickers:
    if tick not in extra_data.columns:
        continue
    tick_rets = extra_data[tick].pct_change().dropna()
    common = sector_rets.index.intersection(tick_rets.index)
    corrs = sector_rets.loc[common].corrwith(tick_rets.loc[common])
    avg_corr = corrs.mean()
    print(f'{tick}: avg_corr={avg_corr:.3f}, min={corrs.min():.3f}, max={corrs.max():.3f}')
    print(f'  {corrs.round(3).to_dict()}')
    print()

# Backtest avec differents univers
universes = {
    'Sectors only (11)': (sector_etfs, data),
    'Sectors + IWM (12)': (sector_etfs + ['IWM'], None),
    'Sectors + EFA (12)': (sector_etfs + ['EFA'], None),
    'Sectors + IWM+EFA (13)': (sector_etfs + ['IWM', 'EFA'], None),
    'Sectors + IWM+EFA+EEM (14)': (sector_etfs + ['IWM', 'EFA', 'EEM'], None),
}

print(f'{'Univers':<38} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}')
print('-' * 70)

universe_results = {}
for name, (universe, _data) in universes.items():
    needed = [t for t in universe if t not in data.columns]
    if needed:
        combined_data = pd.concat([data, extra_data[needed]], axis=1).dropna()
    else:
        combined_data = data.dropna()
    r = backtest_v4(combined_data, universe,
                    rsi_period=14, rsi_threshold=40, rsi_exit=60,
                    num_positions=4, holding_period=15, stop_loss=0.08,
                    start_year=2015)
    universe_results[name] = r
    flag = ' <-- MIEUX' if r['sharpe'] > r_base['sharpe'] else ''
    print(f'{name:<38} {r["sharpe"]:>8.3f} {r["cagr"]*100:>7.1f}% {r["max_dd"]*100:>7.1f}% {r["trades"]:>5}{flag}')


**Verdict H16**: A completer apres execution.

Si la correlation de IWM/EFA avec les sector ETFs est > 0.85,
l'ajout n'apporte pas de diversification significative.
Si le Sharpe augmente, c'est du vrai signal (cycles independants).

## H17: Position sizing par distance au RSI extreme

Un ETF avec RSI=15 est beaucoup plus oversold qu'un ETF avec RSI=38.
On alloue plus de capital aux signaux les plus forts.

Formule: `weight = min_w + (max_w - min_w) * (1 - RSI / threshold)`

Contrainte importante: on evite les poids extremes (>40%) qui augmentent le risque.

In [ ]:
# H17: Position sizing dynamique par distance RSI

def backtest_dynamic_size(data, sector_etfs, rsi_threshold=40,
                          num_positions=4, holding_period=15,
                          stop_loss=0.08, min_weight=0.15, max_weight=0.35,
                          start_year=2015):
    bt_data = data.loc[f'{start_year}-01-01':].copy()
    rsi = pd.DataFrame()
    for etf in sector_etfs:
        rsi[etf] = compute_rsi(data[etf], 14).loc[bt_data.index]
    spy_sma200 = data['SPY'].rolling(200).mean().loc[bt_data.index]
    cash = 100000
    positions = {}
    portfolio = []
    trades = 0
    day_count = 0
    for i in range(len(bt_data)):
        day_count += 1
        to_exit = []
        for etf, pos in positions.items():
            days_held = day_count - pos['entry_day']
            rv = rsi[etf].iloc[i] if pd.notna(rsi[etf].iloc[i]) else 50
            cp = bt_data[etf].iloc[i]
            if stop_loss and (cp / pos['entry_price'] - 1) < -stop_loss:
                to_exit.append(etf)
                continue
            if days_held >= holding_period or rv > 60:
                to_exit.append(etf)
        for etf in to_exit:
            cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
            del positions[etf]
            trades += 1
        if pd.notna(spy_sma200.iloc[i]) and bt_data['SPY'].iloc[i] < spy_sma200.iloc[i]:
            for etf in list(positions.keys()):
                cash += positions[etf]['shares'] * bt_data[etf].iloc[i]
                del positions[etf]
                trades += 1
            portfolio.append(cash)
            continue
        candidates = []
        for etf in sector_etfs:
            if etf in positions:
                continue
            rv = rsi[etf].iloc[i]
            if pd.isna(rv) or rv >= rsi_threshold:
                continue
            candidates.append((etf, rv))
        candidates.sort(key=lambda x: x[1])
        available = num_positions - len(positions)
        total_val = cash + sum(p['shares'] * bt_data[e].iloc[i] for e, p in positions.items())
        for etf, rv in candidates[:available]:
            degree = max(0.0, 1.0 - rv / rsi_threshold)
            weight = min_weight + (max_weight - min_weight) * degree
            invest = total_val * weight
            if invest > cash:
                invest = cash * 0.95
            if invest < 100:
                continue
            shares = invest / bt_data[etf].iloc[i]
            positions[etf] = {'shares': shares, 'entry_day': day_count,
                               'entry_price': bt_data[etf].iloc[i]}
            cash -= invest
            trades += 1
        portfolio.append(cash + sum(p['shares'] * bt_data[e].iloc[i] for e, p in positions.items()))
    final_val = cash + sum(p['shares'] * bt_data[e].iloc[-1] for e, p in positions.items())
    portfolio = pd.Series(portfolio, index=bt_data.index[:len(portfolio)])
    returns = portfolio.pct_change().dropna()
    years = len(portfolio) / 252
    cagr = (final_val / 100000) ** (1/years) - 1 if final_val > 0 else -1
    vol = returns.std() * np.sqrt(252)
    sharpe = (cagr - 0.05) / vol if vol > 0 else 0
    max_dd = (portfolio / portfolio.cummax() - 1).min()
    return {'cagr': cagr, 'sharpe': sharpe, 'max_dd': max_dd,
            'net': (final_val-100000)/100000, 'trades': trades, 'portfolio': portfolio}

print('H17: Position sizing dynamique par distance RSI')
print(f'{'Config':<45} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}')
print('-' * 75)
print(f'{'Flat 25% (v4.0 baseline)':<45} {r_base["sharpe"]:>8.3f} {r_base["cagr"]*100:>7.1f}% {r_base["max_dd"]*100:>7.1f}% {r_base["trades"]:>5}')

dynamic_results = {}
for min_w, max_w in [(0.10, 0.40), (0.15, 0.35), (0.15, 0.40), (0.20, 0.35), (0.20, 0.30)]:
    key = f'dyn min={min_w:.0%} max={max_w:.0%}'
    r = backtest_dynamic_size(data, sector_etfs,
                              min_weight=min_w, max_weight=max_w,
                              num_positions=4, holding_period=15,
                              stop_loss=0.08, start_year=2015)
    dynamic_results[key] = r
    flag = ' <-- MIEUX' if r['sharpe'] > r_base['sharpe'] else ''
    print(f'{key:<45} {r["sharpe"]:>8.3f} {r["cagr"]*100:>7.1f}% {r["max_dd"]*100:>7.1f}% {r["trades"]:>5}{flag}')


**Verdict H17**: A completer apres execution.

Note historique: la dynamique sizing avait ete testee en iteration 3 (H9)
et rejetee car moins robuste que le flat 25%.
Cette iteration reteste avec une plage plus etroite (15-35%).

## Synthese iteration 4 - Meilleure configuration

Recapitulatif de tous les tests de cette iteration.
On identifie la meilleure amelioration et on met a jour main.py en consequence.

In [ ]:
# Synthese: comparer toutes les ameliorations testees en iteration 4
print('SYNTHESE ITERATION 4 - Comparaison des ameliorations')
print(f'{'Config':<50} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Trades':>6}')
print('=' * 80)

print(f'{'v4.0 baseline (RSI14<40, X60, 4pos, 15d, SL8%)':<50} {r_base["sharpe"]:>8.3f} {r_base["cagr"]*100:>7.1f}% {r_base["max_dd"]*100:>7.1f}% {r_base["trades"]:>5}')
print()

best_ee = max(entry_exit_results.items(), key=lambda x: x[1]['sharpe'])
print(f'{'H13+14 best: ' + best_ee[0]:<50} {best_ee[1]["sharpe"]:>8.3f} {best_ee[1]["cagr"]*100:>7.1f}% {best_ee[1]["max_dd"]*100:>7.1f}% {best_ee[1]["trades"]:>5}')

best_bb = max(bb_combo_results.items(), key=lambda x: x[1]['sharpe'])
print(f'{'H15 best: ' + best_bb[0]:<50} {best_bb[1]["sharpe"]:>8.3f} {best_bb[1]["cagr"]*100:>7.1f}% {best_bb[1]["max_dd"]*100:>7.1f}% {best_bb[1]["trades"]:>5}')

best_uni = max(universe_results.items(), key=lambda x: x[1]['sharpe'])
print(f'{'H16 best: ' + best_uni[0]:<50} {best_uni[1]["sharpe"]:>8.3f} {best_uni[1]["cagr"]*100:>7.1f}% {best_uni[1]["max_dd"]*100:>7.1f}% {best_uni[1]["trades"]:>5}')

best_dyn = max(dynamic_results.items(), key=lambda x: x[1]['sharpe'])
print(f'{'H17 best: ' + best_dyn[0]:<50} {best_dyn[1]["sharpe"]:>8.3f} {best_dyn[1]["cagr"]*100:>7.1f}% {best_dyn[1]["max_dd"]*100:>7.1f}% {best_dyn[1]["trades"]:>5}')

all_iter4 = {
    'v4.0 baseline': r_base,
    f'H13+14: {best_ee[0]}': best_ee[1],
    f'H15: {best_bb[0]}': best_bb[1],
    f'H16: {best_uni[0]}': best_uni[1],
    f'H17: {best_dyn[0]}': best_dyn[1],
}
overall_best = max(all_iter4.items(), key=lambda x: x[1]['sharpe'])
print(f'\n==> GAGNANT: {overall_best[0]}')
print(f'    Sharpe: {r_base["sharpe"]:.3f} -> {overall_best[1]["sharpe"]:.3f} ({overall_best[1]["sharpe"]-r_base["sharpe"]:+.3f})')
print(f'    CAGR:   {r_base["cagr"]*100:.1f}% -> {overall_best[1]["cagr"]*100:.1f}%')
print(f'    MaxDD:  {r_base["max_dd"]*100:.1f}% -> {overall_best[1]["max_dd"]*100:.1f}%')


In [ ]:
# Visualisation: equity curves des meilleures configs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

plot_configs = [
    ('v4.0 baseline', r_base),
    (f'H13+14: {best_ee[0]}', best_ee[1]),
    (f'H16: {best_uni[0]}', best_uni[1]),
]

spy_bt = data.loc['2015-01-01':, 'SPY']
spy_norm = spy_bt / spy_bt.iloc[0]

for name, r in plot_configs:
    port = r['portfolio'] / 100000
    ax1.plot(port.index, port.values, label=f'{name} (S={r["sharpe"]:.2f})')
ax1.plot(spy_norm.index, spy_norm.values, '--', alpha=0.5, label='SPY B&H')
ax1.set_title('MeanReversion iter4 - Equity Curves')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

for name, r in plot_configs:
    dd = (r['portfolio'] / r['portfolio'].cummax() - 1) * 100
    ax2.plot(dd.index, dd.values, label=name)
ax2.set_title('Drawdowns (%)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Conclusions et Recommandations Iteration 4

### Verdicts des hypotheses

| Hypothese | Verdict | Detail |
|-----------|---------|--------|
| H12: RSI periods (7,10,14,21) | A VERIFIER | Voir resultats Forward Returns |
| H13: Seuil entree RSI oversold | A VERIFIER | Voir grille H13+H14 |
| H14: Seuil sortie RSI overbought | A VERIFIER | Voir grille H13+H14 |
| H15: Bollinger Bands confirmation | A VERIFIER | Impact sur frequence signaux |
| H16: Multi-asset (IWM/EFA/EEM) | A VERIFIER | Correlations + Sharpe |
| H17: Dynamic sizing RSI distance | A VERIFIER | vs flat 25% |

### Configuration recommandee pour main.py (v5.0)

A completer apres execution:

```python
# Voir resultats ci-dessus pour les valeurs optimales
rsi_period = 14        # H12: stable, evite le bruit de RSI(7)
rsi_oversold = 40      # H13: confirme ou ajuste
rsi_exit = 60          # H14: confirme ou ajuste
num_positions = 4      # Confirme iter3
holding_period = 15    # Confirme iter3
stop_loss = 0.08       # Confirme iter3
universe = sector_etfs  # H16: + IWM/EFA si benefique
bb_confirmation = None  # H15: seulement si Sharpe > baseline et trades > 200
```

### Principe d'integrite
Toute amelioration retenue dans v5.0 doit:
1. Venir du signal mean reversion, pas de l'exposition au marche
2. Garder un nombre de trades raisonnable (>200 sur 10 ans)
3. Etre stable sur plusieurs sous-periodes (2015-18, 2018-21, 2021-25)
4. Ne pas rajouter SPY/QQQ comme position permanente (beta loading interdit)